In [1]:
"""
Weight Sharing — Tiny Recursive Models (HyperNetwork)
======================================================
A HyperNetwork (Ha et al. 2016) is a small auxiliary network that *generates*
the weights of a larger main network. The main network holds no independent
weight parameters for the compressed layers — all weights flow from the tiny
hypernetwork at forward-pass time.

This is the proper "Tiny Recursive Model" for vision: weight generation is
conditioned on a block-index embedding, making it recursive across depth.

METHOD
======
  For each ResNet stage (layer1–4), the conv2 (3×3) weights of every
  bottleneck block are generated by a shared HyperNetwork:

    HyperNet_stage: R^z → R^(C_out × C_in × 3 × 3)

    Input  : block-index embedding  e_i ∈ R^z   (z = EMBEDDING_DIM, default 16)
    Output : full weight tensor     W_i ∈ R^(C_out × C_in × 9)

  Architecture of HyperNet per stage:
    Embedding(num_blocks, z) → Linear(z, hidden) → ReLU → Linear(hidden, C_out×C_in×9)

  hidden = HYPERNET_HIDDEN (default 128)

  Each stage has its own HyperNetwork (different output dimensions).
  Conv1, conv3, BN, downsample remain standard (not compressed).

WHY THIS IS "TINY RECURSIVE"
=============================
  - Tiny    : the HyperNet has far fewer parameters than the weights it generates.
              E.g. for layer3 (256 in, 256 out, 3×3):
                  Original conv2 params per block : 256×256×9 = 589,824
                  HyperNet params (shared)        : z×hidden + hidden×(256×256×9)
                                                  ≈ 16×128 + 128×589824 ≈ 75.5M  ← large
              So for large layers we cap output dim using projection:
              HyperNet → small latent → linear projection to full weight.

  - Recursive: block i's weight is a function of its index embedding e_i,
               and the same HyperNet processes all blocks sequentially →
               implicit depth-wise recursion.

PRACTICAL DESIGN (avoids HyperNet being larger than original)
=============================================================
  For large conv2 layers (many channels), direct weight generation explodes.
  We use a two-stage approach:
    1. HyperNet generates a low-rank factor pair (U_i, V_i) of rank HYPERNET_RANK
    2. Full weight = U_i × V_i  (outer product reconstruction)

  This keeps HyperNet small:
    HyperNet params ≈ z×hidden + hidden×(C_out + C_in×9)×rank
  vs original per-block params: C_out×C_in×9

  When C_out×C_in×9 × rank < original_per_block × num_blocks → compression achieved.

OUTPUT
======
  __4__tiny_recursive.json
"""

import os, json, time, copy, tempfile, warnings, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch : {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
ORIGINAL_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
MODEL_ARCH            = "resnet50"
OUTPUT_JSON           = "__4__tiny_recursive.json"

NUM_CLASSES  = 10
BATCH_SIZE   = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS  = 0
PIN_MEMORY   = DEVICE.type == "cuda"

# HyperNetwork architecture
EMBEDDING_DIM   = 16    # block-index embedding size (z)
HYPERNET_HIDDEN = 128   # hidden layer width in HyperNet MLP
HYPERNET_RANK   = 8     # low-rank factor rank for weight generation

# Fine-tuning
FINETUNE_EPOCHS = 5
FINETUNE_LR     = 1e-3

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] EmbDim={EMBEDDING_DIM}  HiddenDim={HYPERNET_HIDDEN}  "
      f"Rank={HYPERNET_RANK}  FTEpochs={FINETUNE_EPOCHS}", flush=True)


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(arch="resnet50", num_classes=10):
    m = models.resnet50(weights=None) if arch == "resnet50" else models.resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m


def load_model(path, arch):
    print(f"[model] Loading {arch} from {path} ...", flush=True)
    m = build_model(arch, NUM_CLASSES)
    m.load_state_dict(torch.load(path, map_location="cpu"))
    return m.to(DEVICE).eval()


# ── Data ──────────────────────────────────────────────────────────────────────
def get_loaders():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10("../data", train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10("../data", train=False, download=True, transform=test_tf)
    return (
        DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        DataLoader(test_ds,  512,        shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
    )


# ── Helpers ───────────────────────────────────────────────────────────────────
def model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)


def param_count(model):
    return sum(p.numel() for p in model.parameters())


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    return {
        "accuracy" : float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall"   : float(recall_score(labels,    preds, average="macro", zero_division=0)),
        "f1"       : float(f1_score(labels,         preds, average="macro", zero_division=0)),
    }


def measure_inference_ms(model, device, batch_size=128, runs=20):
    m = model.eval().to(device)
    dummy = torch.randn(batch_size, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(3): m(dummy)
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs): m(dummy)
        if device.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1000


# ═══════════════════════════════════════════════════════════════════════════════
# HyperNetwork — generates conv2 weights from block-index embeddings
# ═══════════════════════════════════════════════════════════════════════════════

class StageHyperNetwork(nn.Module):
    """
    Tiny HyperNetwork for one ResNet stage.

    Generates the 3×3 conv2 weight tensor for each block via low-rank factorisation:
      W_i ≈ U_i @ V_i
      U_i ∈ R^(C_out × rank)       — "output" factor
      V_i ∈ R^(rank × C_in×kH×kW) — "input-spatial" factor

    Architecture:
      block_idx → Embedding(n_blocks, z) → Linear(z, H) → ReLU →
        ├─ Linear(H, C_out × rank)         → reshape → U_i
        └─ Linear(H, rank × C_in×kH×kW)   → reshape → V_i

    W_i = U_i @ V_i  →  shape (C_out, C_in×kH×kW) → reshape (C_out, C_in, kH, kW)

    Parameters:
      n_blocks      : number of blocks in this stage
      C_out, C_in   : conv2 channel dimensions
      kH, kW        : kernel size (3, 3)
      embed_dim     : block embedding size z
      hidden        : MLP hidden width H
      rank          : low-rank approximation rank r
    """
    def __init__(self, n_blocks, C_out, C_in, kH, kW,
                 embed_dim=EMBEDDING_DIM, hidden=HYPERNET_HIDDEN, rank=HYPERNET_RANK):
        super().__init__()
        self.n_blocks = n_blocks
        self.C_out    = C_out
        self.C_in     = C_in
        self.kH       = kH
        self.kW       = kW
        self.rank     = rank
        self.weight_shape = (C_out, C_in, kH, kW)
        D = C_in * kH * kW   # flattened spatial-channel dim

        # Block index embedding table
        self.embedding = nn.Embedding(n_blocks, embed_dim)

        # Shared MLP trunk
        self.trunk = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
        )

        # Two heads: U and V factors
        self.head_U = nn.Linear(hidden, C_out * rank)    # → (C_out, rank)
        self.head_V = nn.Linear(hidden, rank * D)         # → (rank, D)

        # Initialise heads with small weights to keep generated weights near zero at start
        nn.init.normal_(self.head_U.weight, std=0.01)
        nn.init.normal_(self.head_V.weight, std=0.01)
        nn.init.zeros_(self.head_U.bias)
        nn.init.zeros_(self.head_V.bias)

    def generate_weight(self, block_idx: int) -> torch.Tensor:
        """Generate conv2 weight tensor for block `block_idx`."""
        idx = torch.tensor([block_idx], device=self.embedding.weight.device)
        e   = self.embedding(idx)          # (1, embed_dim)
        h   = self.trunk(e)                # (1, hidden)

        U = self.head_U(h).reshape(self.C_out, self.rank)        # (C_out, rank)
        V = self.head_V(h).reshape(self.rank, self.C_in * self.kH * self.kW)  # (rank, D)

        W = (U @ V).reshape(self.weight_shape)   # (C_out, C_in, kH, kW)
        return W

    def hypernet_param_count(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def original_param_count(self) -> int:
        """How many params the original conv2 tensors occupied across all blocks."""
        return self.C_out * self.C_in * self.kH * self.kW * self.n_blocks


class HyperBottleneck(nn.Module):
    """
    Wraps a ResNet Bottleneck block to use a shared StageHyperNetwork for conv2.
    At each forward pass, the HyperNet generates the conv2 weight on-the-fly.
    conv1, conv3, BN, downsample remain standard (independent parameters).
    """
    def __init__(self, original_block: nn.Module,
                 hypernet: StageHyperNetwork, block_idx: int):
        super().__init__()
        self.conv1      = original_block.conv1
        self.bn1        = original_block.bn1
        self.hypernet   = hypernet        # shared across all blocks in this stage
        self.block_idx  = block_idx
        self.bn2        = original_block.bn2
        self.conv3      = original_block.conv3
        self.bn3        = original_block.bn3
        self.relu       = original_block.relu
        self.downsample = original_block.downsample

        # Store conv2 metadata for F.conv2d call
        orig_conv2 = original_block.conv2
        self.stride   = orig_conv2.stride
        self.padding  = orig_conv2.padding
        self.dilation = orig_conv2.dilation
        self.groups   = orig_conv2.groups
        # bias is not used in ResNet conv layers (BN follows)
        self.use_bias = orig_conv2.bias is not None
        if self.use_bias:
            self.conv2_bias = nn.Parameter(orig_conv2.bias.clone())

    def forward(self, x):
        identity = x

        # conv1 → bn1 → relu
        out = self.relu(self.bn1(self.conv1(x)))

        # conv2 — weight generated by HyperNet on the fly
        W = self.hypernet.generate_weight(self.block_idx)
        bias = self.conv2_bias if self.use_bias else None
        out = F.conv2d(out, W, bias,
                       stride=self.stride, padding=self.padding,
                       dilation=self.dilation, groups=self.groups)
        out = self.relu(self.bn2(out))

        # conv3 → bn3
        out = self.bn3(self.conv3(out))

        # Residual connection
        if self.downsample is not None:
            identity = self.downsample(x)
        out = self.relu(out + identity)
        return out


# ── Apply HyperNetwork ────────────────────────────────────────────────────────
def apply_hypernetwork(model: nn.Module) -> tuple:
    """
    For each ResNet stage (layer1–4):
      1. Build a StageHyperNetwork sized for that stage's conv2 dimensions
      2. Initialise HyperNet so it initially reproduces the original weights
         (via least-squares low-rank init)
      3. Replace each block with HyperBottleneck

    Returns (hyper_model, info_dict)
    """
    model = copy.deepcopy(model)
    info  = {}

    for stage_name in ["layer1", "layer2", "layer3", "layer4"]:
        stage  = getattr(model, stage_name)
        blocks = list(stage.children())

        # Identify conv2 layers
        conv2s = [
            b.conv2 for b in blocks
            if hasattr(b, "conv2") and isinstance(b.conv2, nn.Conv2d)
        ]
        if len(conv2s) < 2:
            print(f"  [hyper] {stage_name}: too few blocks, skipping", flush=True)
            continue

        ref_shape = conv2s[0].weight.shape
        if not all(c.weight.shape == ref_shape for c in conv2s):
            print(f"  [hyper] {stage_name}: shape mismatch, skipping", flush=True)
            continue

        C_out, C_in, kH, kW = ref_shape
        n_blocks = len(conv2s)

        # Build HyperNetwork for this stage
        hypernet = StageHyperNetwork(
            n_blocks=n_blocks, C_out=C_out, C_in=C_in, kH=kH, kW=kW,
            embed_dim=EMBEDDING_DIM, hidden=HYPERNET_HIDDEN, rank=HYPERNET_RANK,
        ).to(DEVICE)

        # Warm-start HyperNet: minimise ||W_generated_i - W_original_i||
        # for each block via a quick SGD pass on the embedding + heads only.
        _warm_start_hypernet(hypernet, conv2s, n_blocks, C_out, C_in, kH, kW)

        # Wrap each block with HyperBottleneck
        new_blocks = nn.Sequential(*[
            HyperBottleneck(block, hypernet, idx)
            for idx, block in enumerate(blocks)
        ])
        setattr(model, stage_name, new_blocks)

        hyper_params    = hypernet.hypernet_param_count()
        original_params = hypernet.original_param_count()
        reduction       = 1.0 - hyper_params / original_params

        info[stage_name] = {
            "num_blocks"        : n_blocks,
            "conv2_shape"       : list(ref_shape),
            "hypernet_params"   : hyper_params,
            "original_params"   : original_params,
            "param_reduction_pct": round(reduction * 100, 2),
            "embedding_dim"     : EMBEDDING_DIM,
            "hidden_dim"        : HYPERNET_HIDDEN,
            "rank"              : HYPERNET_RANK,
        }
        print(f"  [hyper] {stage_name}: {n_blocks} blocks, conv2{tuple(ref_shape)}, "
              f"HyperNet params={hyper_params:,} vs original={original_params:,} "
              f"({reduction*100:.1f}% reduction)", flush=True)

    return model, info


def _warm_start_hypernet(hypernet: StageHyperNetwork,
                          conv2s: list, n_blocks: int,
                          C_out: int, C_in: int, kH: int, kW: int,
                          steps: int = 200, lr: float = 1e-2):
    """
    Quick optimisation to make generated weights approximate the original conv2 weights.
    This avoids random initialisation causing a large accuracy drop before fine-tuning.
    Uses Adam on the HyperNet parameters only.
    """
    targets = [c.weight.data.to(DEVICE) for c in conv2s]
    optimizer = torch.optim.Adam(hypernet.parameters(), lr=lr)

    hypernet.train()
    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)
        loss = torch.tensor(0.0, device=DEVICE)
        for i in range(n_blocks):
            W_gen = hypernet.generate_weight(i)
            loss  = loss + F.mse_loss(W_gen, targets[i])
        loss = loss / n_blocks
        loss.backward()
        optimizer.step()

    hypernet.eval()
    # Report final reconstruction error
    with torch.no_grad():
        final_loss = sum(
            F.mse_loss(hypernet.generate_weight(i), targets[i]).item()
            for i in range(n_blocks)
        ) / n_blocks
    print(f"      Warm-start done ({steps} steps), avg MSE={final_loss:.6f}", flush=True)


# ── Fine-tuning ───────────────────────────────────────────────────────────────
def finetune(model, train_loader, epochs, lr):
    if epochs == 0:
        return []
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    accs = []
    print(f"\n  Fine-tuning {epochs} epochs @ lr={lr} ...", flush=True)
    for epoch in range(epochs):
        correct = total = 0
        for inputs, targets in train_loader:
            inputs  = inputs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                loss = F.cross_entropy(model(inputs), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            with torch.no_grad():
                preds    = model(inputs).argmax(1)
                correct += (preds == targets).sum().item()
                total   += targets.size(0)
        scheduler.step()
        acc = correct / total
        accs.append(round(acc, 6))
        print(f"    Epoch {epoch+1}/{epochs}  train_acc={acc:.4f}", flush=True)
    return accs


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Weight Sharing — Tiny Recursive Models (HyperNetwork)")
    print(f"  Model   : {MODEL_ARCH}  |  Device: {DEVICE}")
    print(f"  HyperNet: embed={EMBEDDING_DIM}  hidden={HYPERNET_HIDDEN}  rank={HYPERNET_RANK}")
    print(f"  FineTune: {FINETUNE_EPOCHS} epochs @ lr={FINETUNE_LR}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline accuracy : {baseline_acc:.4f}\n", flush=True)

    original_model   = load_model(ORIGINAL_MODEL_PATH, MODEL_ARCH)
    original_size_mb = model_size_mb(original_model)
    original_params  = param_count(original_model)
    print(f"  Original size     : {original_size_mb:.2f} MB", flush=True)
    print(f"  Original params   : {original_params:,}\n", flush=True)

    train_loader, test_loader = get_loaders()

    # ── Apply HyperNetwork ────────────────────────────────────────────────────
    t0 = time.perf_counter()
    print("  Building HyperNetworks and warm-starting ...", flush=True)
    hyper_model, stage_info = apply_hypernetwork(original_model)
    hyper_model   = hyper_model.to(DEVICE)
    compress_time = time.perf_counter() - t0
    print(f"  Done in {compress_time:.1f}s", flush=True)

    compressed_params = param_count(hyper_model)
    param_reduction   = 1.0 - compressed_params / original_params
    print(f"  Compressed params : {compressed_params:,}  "
          f"({param_reduction*100:.1f}% reduction)\n", flush=True)

    # Pre-finetune evaluation
    pre_ft = evaluate(hyper_model, test_loader)
    print(f"  Pre-FT accuracy   : {pre_ft['accuracy']:.4f}", flush=True)

    # Fine-tuning (end-to-end: both HyperNet and remaining standard layers)
    ft_accs = finetune(hyper_model, train_loader, FINETUNE_EPOCHS, FINETUNE_LR)

    # Final evaluation
    metrics  = evaluate(hyper_model, test_loader)
    infer_ms = measure_inference_ms(hyper_model, DEVICE)
    size_mb  = model_size_mb(hyper_model)
    acc_drop = baseline_acc - metrics["accuracy"]

    print(f"\n  ✓ Results:")
    print(f"    Accuracy         : {metrics['accuracy']:.4f}  (drop={acc_drop:+.4f})")
    print(f"    Size             : {size_mb:.2f} MB  (original={original_size_mb:.2f} MB)")
    print(f"    Compression ratio: {original_size_mb/size_mb:.2f}×")
    print(f"    Param reduction  : {param_reduction*100:.1f}%")
    print(f"    Inference        : {infer_ms:.1f} ms")

    report = {
        "experiment"             : "weight_sharing_tiny_recursive",
        "method"                 : "weight_sharing",
        "variant"                : "TinyRecursiveHyperNetwork",
        "original_arch"          : MODEL_ARCH,
        "dataset"                : "CIFAR-10",
        "train_device"           : str(DEVICE),
        # hyperparameters
        "embedding_dim"          : EMBEDDING_DIM,
        "hypernet_hidden"        : HYPERNET_HIDDEN,
        "hypernet_rank"          : HYPERNET_RANK,
        "finetune_epochs"        : FINETUNE_EPOCHS,
        "finetune_lr"            : FINETUNE_LR,
        # method description
        "method_description"     : (
            f"HyperNetwork (Ha et al. 2016) for ResNet50: a small MLP per stage maps "
            f"block-index embeddings (dim={EMBEDDING_DIM}) → low-rank weight factors "
            f"(rank={HYPERNET_RANK}) via a hidden layer (dim={HYPERNET_HIDDEN}). "
            f"W_i = U_i @ V_i where U_i∈R^(C_out×r), V_i∈R^(r×C_in·kH·kW). "
            f"Warm-started (200 SGD steps, MSE loss) before fine-tuning. "
            f"Conv1, conv3, BN, downsample remain standard independent parameters."
        ),
        "stage_info"             : stage_info,
        # timing
        "compression_time_s"     : round(compress_time, 2),
        # baseline
        "baseline"               : baseline,
        "original_size_mb"       : round(original_size_mb, 4),
        "original_params"        : original_params,
        # performance
        "pre_finetune_accuracy"  : round(pre_ft["accuracy"], 6),
        "accuracy"               : round(metrics["accuracy"],  6),
        "precision"              : round(metrics["precision"], 6),
        "recall"                 : round(metrics["recall"],    6),
        "f1"                     : round(metrics["f1"],        6),
        "accuracy_drop"          : round(acc_drop, 6),
        "finetune_epoch_accs"    : ft_accs,
        # size & efficiency
        "compressed_params"      : compressed_params,
        "param_reduction_pct"    : round(param_reduction * 100, 2),
        "size_disk_mb"           : round(size_mb, 4),
        "compression_ratio"      : round(original_size_mb / size_mb, 4) if size_mb else None,
        "inference_ms"           : round(infer_ms, 4),
        "status"                 : "ok",
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  ✓ Saved → {OUTPUT_JSON}")


if __name__ == "__main__":
    main()

[init] PyTorch : 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] EmbDim=16  HiddenDim=128  Rank=8  FTEpochs=5

  Weight Sharing — Tiny Recursive Models (HyperNetwork)
  Model   : resnet50  |  Device: cuda
  HyperNet: embed=16  hidden=128  rank=8
  FineTune: 5 epochs @ lr=0.001

  Baseline accuracy : 0.9320

[model] Loading resnet50 from ../__2__baseline_resnet50_cifar10.pth ...
  Original size     : 90.03 MB
  Original params   : 23,520,842

  Building HyperNetworks and warm-starting ...
      Warm-start done (200 steps), avg MSE=0.000021
  [hyper] layer1: 3 blocks, conv2(64, 64, 3, 3), HyperNet params=679,216 vs original=110,592 (-514.2% reduction)
      Warm-start done (200 steps), avg MSE=0.000013
  [hyper] layer2: 4 blocks, conv2(128, 128, 3, 3), HyperNet params=1,339,712 vs original=589,824 (-127.1% reduction)
      Warm-start done (200 steps), avg MSE=0.000018
  [hyper] layer3: 6 blocks, conv2(256, 256, 3, 3), HyperNet params=2,660,704 vs original=3,538,944 (24.8% reduction